# OccamRoute — Notebook 02: Resolução de Tarefas Reais e Mensuração End-to-End

> **Missão:** Colocar o algoritmo completo (**AST RepoGraph + Recorte Cirúrgico + Roteador ML treinado no Notebook 01**) para **resolver tarefas reais** de engenharia de software no Google Colab (GPU T4/L4 ≤ 15 GB VRAM) e **mensurar** rigorosamente 4 pipelines controlados.

## Protocolo Experimental
1. **Alvo Real:** Tarefas reais do `princeton-nlp/SWE-bench_Lite` (com fallback automático para suíte `encode/httpx@0.28.1` caso offline) onde cada hunk de patch oficial é mapeado deterministicamente para os nós AST correspondentes no `RepoGraph`.
2. **4 Pipelines Comparados:**
   - **Baseline A (`Brute Force`)**: arquivos inteiros do módulo (expõe *Lost in the Middle*).
   - **Baseline B (`Naive RAG`)**: chunks de 512 tokens + `BAAI/bge-small-en-v1.5` + FAISS Top-5.
   - **Baseline C (`Unbounded AST 2-Hop`)**: subgrafo 2-hop sem orçamento de tokens (efeito *RepoGraph* sem poda).
   - **OccamRoute (`Slicer + ML Router`)**: subgrafo limitado por orçamento + *Position-Aware Ordering* + `OccamRouter` (PyTorch MLP / Sklearn GBDT) roteando entre GPU local 4-bit e API de fronteira.
3. **Métricas:** `File Recall`, `Function/Hunk Recall`, `Symbol F1`, `Middle-Burial (%)`, `Context Tokens`, `Latency (ms)`, `Cost / 1,000 Tasks` e **RouteLLM PGR (%)**.

In [ ]:
# @title 1. Verificação de Hardware Colab (VRAM <= 15 GB) e Imports do `occamroute`
import os
import platform
import sys
from pathlib import Path

!pip install -q datasets networkx numpy scipy tiktoken faiss-cpu sentence-transformers transformers accelerate bitsandbytes google-genai pandas scikit-learn joblib torch matplotlib seaborn

for candidate in [
    Path("src"),
    Path("../src"),
    Path("/content/OccamRoute/src"),
    Path("/content/src"),
]:
    if (candidate / "occamroute").exists() and str(candidate.resolve()) not in sys.path:
        sys.path.insert(0, str(candidate.resolve()))
        break

import torch
from occamroute import (
    FEATURE_NAMES,
    OccamRouter,
    RepoGraph,
    RepoGraphBuilder,
    SklearnComplexityRouter,
    SubgraphFeatureExtractor,
    SurgicalContextSlicer,
    TorchComplexityRouter,
    build_routing_dataset,
    count_tokens,
    load_swebench_lite,
)

print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU     : {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.2f} GB")
else:
    print("GPU     : CPU mode (inferência usará solver calibrado).")

In [ ]:
# @title 2. Carregar Roteadores Treinados (`torch_router.pt` / `sklearn_router.joblib`)
import json

artifacts_dir = Path("artifacts")
pt_artifact = artifacts_dir / "torch_router.pt"
sk_artifact = artifacts_dir / "sklearn_router.joblib"
metrics_file = artifacts_dir / "router_metrics.json"

if not pt_artifact.exists() or not sk_artifact.exists():
    print("Artefatos não encontrados; treinando roteadores rapidamente...")
    pkg_dir = next(
        (p.resolve() for p in [Path("src/occamroute"), Path("../src/occamroute"), Path("/content/OccamRoute/src/occamroute")] if p.exists()),
    )
    boot_graph = RepoGraphBuilder().build_from_directory(pkg_dir)
    boot_ds = build_routing_dataset(repo_graph=boot_graph, n_samples=420, val_ratio=0.25, seed=42)
    SklearnComplexityRouter().fit(boot_ds.X_train, boot_ds.y_train).save(sk_artifact)
    TorchComplexityRouter().fit(boot_ds.X_train, dataset_y := boot_ds.y_train).save(pt_artifact)

optimal_tau = 0.50
if metrics_file.exists():
    optimal_tau = float(json.loads(metrics_file.read_text(encoding="utf-8")).get("optimal_tau", 0.50))

pt_router = OccamRouter.from_artifact(pt_artifact, threshold=optimal_tau)
sk_router = OccamRouter.from_artifact(sk_artifact, threshold=optimal_tau)
print(f"Roteadores carregados | tau={optimal_tau:.2f} | PyTorch={pt_artifact} | Sklearn={sk_artifact}")

In [ ]:
# @title 3. Tarefas Reais (`SWE-bench Lite` + Mapeamento Automático de Patch Hunks -> Símbolos AST)
import subprocess
from dataclasses import dataclass
from typing import Literal


@dataclass(frozen=True)
class EvalTask:
    task_id: str
    repo: str
    expected_complexity: Literal["low", "high"]
    prompt: str
    expected_files: list[str]
    expected_symbols: list[str]
    target_module_files: list[str]


def map_patch_hunks_to_ast_symbols(
    repo_graph: RepoGraph,
    gt_hunks: dict[str, list[tuple[int, int]]],
) -> list[str]:
    """Mapeia intervalos de linhas de um patch SWE-bench para os nós AST exatos do RepoGraph."""
    matched: list[str] = []
    for file_path, ranges in gt_hunks.items():
        for nid in repo_graph.file_to_nodes.get(file_path, []):
            if nid not in repo_graph.leaf_ids:
                continue
            node = repo_graph.nodes[nid]
            for h_start, h_end in ranges:
                if max(node.start_line, h_start) <= min(node.end_line, h_end):
                    if nid not in matched:
                        matched.append(nid)
                    break
    return matched


# Tenta carregar tarefas reais do `psf/requests` via SWE-bench Lite; fallback limpo para `encode/httpx@0.28.1`
USE_SWEBENCH_REQUESTS = True
EVAL_REPO_URL = "https://github.com/psf/requests.git"
EVAL_REPO_DIR = Path("/content/eval_requests") if Path("/content").exists() else Path("./_bench_repo/requests")
EVAL_TASKS: list[EvalTask] = []

try:
    if USE_SWEBENCH_REQUESTS:
        swe_req = load_swebench_lite(limit=8, repos=["psf/requests"])
        if swe_req:
            if not EVAL_REPO_DIR.exists():
                EVAL_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
                subprocess.run(["git", "clone", "--quiet", EVAL_REPO_URL, str(EVAL_REPO_DIR)], check=True)
            # Checkout do commit base da primeira tarefa e mapeia todas que existem nesse tree ou faz checkout por tarefa
            for t in swe_req:
                subprocess.run(["git", "-C", str(EVAL_REPO_DIR), "checkout", "--quiet", t.base_commit], check=False)
                g_tmp = RepoGraphBuilder().build_from_directory(EVAL_REPO_DIR / "src" if (EVAL_REPO_DIR / "src").exists() else EVAL_REPO_DIR)
                rel_prefix = "src/" if (EVAL_REPO_DIR / "src").exists() else ""
                adj_hunks = {
                    (k[len(rel_prefix):] if rel_prefix and k.startswith(rel_prefix) else k): v
                    for k, v in t.ground_truth_hunks.items()
                }
                syms = map_patch_hunks_to_ast_symbols(g_tmp, adj_hunks)
                adj_files = [
                    (f[len(rel_prefix):] if rel_prefix and f.startswith(rel_prefix) else f)
                    for f in t.ground_truth_files
                    if (f[len(rel_prefix):] if rel_prefix and f.startswith(rel_prefix) else f) in g_tmp.file_to_nodes
                ]
                if syms and adj_files:
                    churn = t.patch_added_lines + t.patch_removed_lines
                    comp: Literal["low", "high"] = "high" if (len(adj_files) > 1 or churn >= 12 or len(syms) >= 3) else "low"
                    EVAL_TASKS.append(
                        EvalTask(
                            task_id=t.instance_id,
                            repo=t.repo,
                            expected_complexity=comp,
                            prompt=t.problem_statement[:900],
                            expected_files=adj_files,
                            expected_symbols=syms,
                            target_module_files=adj_files,
                        )
                    )
            if EVAL_TASKS:
                repo_graph = RepoGraphBuilder().build_from_directory(EVAL_REPO_DIR / "src" if (EVAL_REPO_DIR / "src").exists() else EVAL_REPO_DIR)
                print(f"SWE-bench Lite (`psf/requests`): {len(EVAL_TASKS)} tarefas reais mapeadas ao AST.")
except Exception as exc:
    print(f"Fallback para `encode/httpx@0.28.1` ({exc}).")

if not EVAL_TASKS:
    EVAL_REPO_URL = "https://github.com/encode/httpx.git"
    EVAL_REPO_DIR = Path("/content/httpx") if Path("/content").exists() else Path("./_bench_repo/httpx")
    if not (EVAL_REPO_DIR / "httpx").exists():
        EVAL_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "0.28.1", EVAL_REPO_URL, str(EVAL_REPO_DIR)], check=True)
    repo_graph = RepoGraphBuilder().build_from_directory(EVAL_REPO_DIR)
    EVAL_TASKS = [
        EvalTask("T01", "encode/httpx", "low", "Explique o contrato de `codes.is_redirect` e `codes.get_reason_phrase`.", ["httpx/_status_codes.py"], ["httpx/_status_codes.py::codes.is_redirect", "httpx/_status_codes.py::codes.get_reason_phrase"], ["httpx/_status_codes.py", "httpx/_models.py"]),
        EvalTask("T02", "encode/httpx", "low", "Como `BasicAuth._build_auth_header` e `BasicAuth.auth_flow` funcionam?", ["httpx/_auth.py"], ["httpx/_auth.py::BasicAuth._build_auth_header", "httpx/_auth.py::BasicAuth.auth_flow"], ["httpx/_auth.py"]),
        EvalTask("T03", "encode/httpx", "low", "Quais validações `Timeout.__init__` aplica e o que `Timeout.as_dict` retorna?", ["httpx/_config.py"], ["httpx/_config.py::Timeout.__init__", "httpx/_config.py::Timeout.as_dict"], ["httpx/_config.py", "httpx/_client.py"]),
        EvalTask("T04", "encode/httpx", "low", "Como `Headers.get_list` e `_normalize_header_key` tratam headers?", ["httpx/_models.py"], ["httpx/_models.py::Headers.get_list", "httpx/_models.py::_normalize_header_key"], ["httpx/_models.py"]),
        EvalTask("T05", "encode/httpx", "low", "Em `Response.raise_for_status`, quando `HTTPStatusError` é levantada e o que `HTTPStatusError.__init__` salva?", ["httpx/_models.py", "httpx/_exceptions.py"], ["httpx/_models.py::Response.raise_for_status", "httpx/_exceptions.py::HTTPStatusError.__init__"], ["httpx/_models.py", "httpx/_exceptions.py"]),
        EvalTask("T06", "encode/httpx", "low", "Explique `URLPattern.matches` e `URLPattern.priority`.", ["httpx/_utils.py"], ["httpx/_utils.py::URLPattern.matches", "httpx/_utils.py::URLPattern.priority"], ["httpx/_utils.py", "httpx/_config.py"]),
        EvalTask("T07", "encode/httpx", "high", "Investigue vazamento em redirects cross-origin: `BaseClient._redirect_headers`, `_same_origin` e `_is_https_redirect`.", ["httpx/_client.py"], ["httpx/_client.py::BaseClient._redirect_headers", "httpx/_client.py::_same_origin", "httpx/_client.py::_is_https_redirect"], ["httpx/_client.py", "httpx/_models.py"]),
        EvalTask("T08", "encode/httpx", "high", "Rastreie `Client.send` -> `Client._send_handling_auth` -> `Client._send_handling_redirects` -> `Client._send_single_request` -> `HTTPTransport.handle_request`.", ["httpx/_client.py", "httpx/_transports/default.py"], ["httpx/_client.py::Client.send", "httpx/_client.py::Client._send_handling_auth", "httpx/_client.py::Client._send_handling_redirects", "httpx/_client.py::Client._send_single_request", "httpx/_transports/default.py::HTTPTransport.handle_request"], ["httpx/_client.py", "httpx/_transports/default.py"]),
        EvalTask("T09", "encode/httpx", "high", "Analise o handshake `DigestAuth.auth_flow`, `DigestAuth._parse_challenge`, `DigestAuth._resolve_qop` e `DigestAuth._build_auth_header`.", ["httpx/_auth.py"], ["httpx/_auth.py::DigestAuth.auth_flow", "httpx/_auth.py::DigestAuth._parse_challenge", "httpx/_auth.py::DigestAuth._build_auth_header", "httpx/_auth.py::DigestAuth._resolve_qop"], ["httpx/_auth.py", "httpx/_client.py"]),
        EvalTask("T10", "encode/httpx", "high", "Proponha refatoração de `Client._send_handling_redirects` e `AsyncClient._send_handling_redirects` preservando `BaseClient._build_redirect_request` e `TooManyRedirects`.", ["httpx/_client.py", "httpx/_exceptions.py"], ["httpx/_client.py::Client._send_handling_redirects", "httpx/_client.py::AsyncClient._send_handling_redirects", "httpx/_client.py::BaseClient._build_redirect_request", "httpx/_exceptions.py::TooManyRedirects"], ["httpx/_client.py", "httpx/_exceptions.py"]),
        EvalTask("T11", "encode/httpx", "high", "Rastreie descompressão streaming: `Response._get_content_decoder`, `Response.iter_bytes`, `MultiDecoder.decode` e `GZipDecoder.decode`.", ["httpx/_models.py", "httpx/_decoders.py"], ["httpx/_models.py::Response._get_content_decoder", "httpx/_models.py::Response.iter_bytes", "httpx/_decoders.py::MultiDecoder.decode", "httpx/_decoders.py::GZipDecoder.decode"], ["httpx/_models.py", "httpx/_decoders.py"]),
        EvalTask("T12", "encode/httpx", "high", "Rastreie mapeamento `httpcore` -> `httpx`: `AsyncHTTPTransport.handle_async_request`, `map_httpcore_exceptions`, `_load_httpcore_exceptions` e `request_context`.", ["httpx/_transports/default.py", "httpx/_exceptions.py"], ["httpx/_transports/default.py::map_httpcore_exceptions", "httpx/_transports/default.py::_load_httpcore_exceptions", "httpx/_transports/default.py::AsyncHTTPTransport.handle_async_request", "httpx/_exceptions.py::request_context"], ["httpx/_transports/default.py", "httpx/_exceptions.py", "httpx/_client.py"]),
    ]

for et in EVAL_TASKS:
    for sym in et.expected_symbols:
        assert sym in repo_graph.nodes, f"Símbolo ausente no RepoGraph: {sym}"

print("RepoGraph Stats:", repo_graph.summary_stats())
print(f"Tarefas prontas para resolução: {len(EVAL_TASKS)}")

In [ ]:
# @title 4. Instanciar os 4 Pipelines Controlados (Brute Force, Naive RAG, Unbounded AST, OccamRoute)
import time
import numpy as np
from dataclasses import dataclass

slicer = SurgicalContextSlicer(repo_graph)
extractor = SubgraphFeatureExtractor(repo_graph)


@dataclass
class ContextBundle:
    context_text: str
    retrieved_symbols: list[str]
    retrieved_files: list[str]
    context_tokens: int
    prep_ms: float


def run_brute_force(task: EvalTask) -> ContextBundle:
    t0 = time.perf_counter()
    blocks, syms = [], []
    for rel in task.target_module_files:
        txt = (repo_graph.root_dir / rel).read_text(encoding="utf-8")
        blocks.append(f"# FILE: {rel}\n{txt}")
        syms.extend(nid for nid in repo_graph.file_to_nodes.get(rel, []) if nid in repo_graph.leaf_ids)
    ctx = "\n\n".join(blocks)
    return ContextBundle(ctx, syms, list(task.target_module_files), count_tokens(ctx), (time.perf_counter() - t0) * 1000.0)


def run_unbounded_ast(task: EvalTask) -> ContextBundle:
    t0 = time.perf_counter()
    scores, _, _, _ = slicer._score_seeds(task.prompt)
    seeds = [sid for _, sid in scores[:3]]
    dist = repo_graph.k_hop_neighbors(seeds, k=2, undirected=True)
    nodes = sorted((nid for nid in dist if nid in repo_graph.leaf_ids), key=lambda s: (repo_graph.nodes[s].file_path, repo_graph.nodes[s].start_line))
    ctx = "\n\n".join(
        f"# FILE: {repo_graph.nodes[s].file_path} | SYMBOL: {repo_graph.nodes[s].qualname}\n{repo_graph.nodes[s].source}"
        for s in nodes
    )
    files = sorted({repo_graph.nodes[s].file_path for s in nodes})
    return ContextBundle(ctx, nodes, files, count_tokens(ctx), (time.perf_counter() - t0) * 1000.0)


class NaiveRAGRetriever:
    def __init__(self, graph: RepoGraph, chunk_tokens: int = 512, overlap: int = 64) -> None:
        self.graph = graph
        self.chunks: list[tuple[str, int, int, str]] = []
        for rel in sorted(graph.file_to_nodes):
            fp = graph.root_dir / rel
            if not fp.exists():
                continue
            lines = fp.read_text(encoding="utf-8").splitlines()
            ltoks = [max(1, count_tokens(ln)) for ln in lines]
            i = 0
            while i < len(lines):
                acc, j = 0, i
                while j < len(lines) and (acc + ltoks[j] <= chunk_tokens or j == i):
                    acc += ltoks[j]
                    j += 1
                self.chunks.append((rel, i + 1, j, "\n".join(lines[i:j])))
                if j >= len(lines):
                    break
                back, ni = 0, j
                while ni > i + 1 and back < overlap:
                    ni -= 1
                    back += ltoks[ni]
                i = max(i + 1, ni)

        self._use_faiss = False
        try:
            import faiss
            from sentence_transformers import SentenceTransformer

            dev = "cuda" if torch.cuda.is_available() else "cpu"
            self.embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=dev)
            embs = np.asarray(self.embedder.encode([f"{c[0]}\n{c[3]}" for c in self.chunks], normalize_embeddings=True, show_progress_bar=False), dtype="float32")
            self.index = faiss.IndexFlatIP(embs.shape[1])
            self.index.add(embs)
            self._use_faiss = True
        except Exception:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.vec = TfidfVectorizer(max_features=4096)
            self.tfidf_mat = self.vec.fit_transform([f"{c[0]}\n{c[3]}" for c in self.chunks])

    def retrieve(self, query: str, top_k: int = 5) -> ContextBundle:
        t0 = time.perf_counter()
        if self._use_faiss:
            qe = np.asarray(self.embedder.encode([query], normalize_embeddings=True, show_progress_bar=False), dtype="float32")
            _, idxs = self.index.search(qe, top_k)
            top_indices = [int(i) for i in idxs[0] if i >= 0]
        else:
            qv = self.vec.transform([query])
            sims = (self.tfidf_mat @ qv.T).toarray().ravel()
            top_indices = [int(i) for i in np.argsort(-sims)[:top_k]]

        blocks, syms, files = [], [], set()
        for idx in top_indices:
            rel, s_ln, e_ln, txt = self.chunks[idx]
            blocks.append(f"# CHUNK: {rel} (L{s_ln}-L{e_ln})\n{txt}")
            files.add(rel)
            for nid in self.graph.file_to_nodes.get(rel, []):
                if nid not in self.graph.leaf_ids:
                    continue
                nd = self.graph.nodes[nid]
                ov = min(e_ln, nd.end_line) - max(s_ln, nd.start_line) + 1
                if ov > 0 and (ov / max(1, nd.end_line - nd.start_line + 1) >= 0.35 or (s_ln <= nd.start_line <= e_ln and ov >= 3)):
                    if nid not in syms:
                        syms.append(nid)
        ctx = "\n\n".join(blocks)
        return ContextBundle(ctx, syms, sorted(files), count_tokens(ctx), (time.perf_counter() - t0) * 1000.0)


rag_retriever = NaiveRAGRetriever(repo_graph)
print(f"Pipelines prontos | NaiveRAG chunks={len(rag_retriever.chunks)} (FAISS={rag_retriever._use_faiss})")

In [ ]:
# @title 5. Backends de Resolução (Local 4-bit GPU <= 15 GB VRAM + Frontier Gemini API)
def _detect_gemini_key() -> str | None:
    if os.environ.get("GEMINI_API_KEY"):
        return os.environ["GEMINI_API_KEY"]
    try:
        from google.colab import userdata  # type: ignore
        sec = userdata.get("GEMINI_API_KEY")
        if sec:
            os.environ["GEMINI_API_KEY"] = sec
            return sec
    except Exception:
        pass
    return None


HAS_GPU = torch.cuda.is_available() and (torch.cuda.get_device_properties(0).total_memory / (1024**3) >= 12.0)
HAS_GEMINI = _detect_gemini_key() is not None

SYSTEM_PROMPT = (
    "You are a Staff Python Systems Engineer. Localize the root cause and propose a minimal fix "
    "citing exact file paths and symbol names from the context."
)


class SolverBackend:
    def __init__(self, tier: str, live: bool) -> None:
        self.tier = tier
        self.live = live
        self.model = None
        self.tokenizer = None
        self.client = None
        if live and tier == "weak_local":
            from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
            bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
            mid = "Qwen/Qwen2.5-Coder-7B-Instruct"
            self.tokenizer = AutoTokenizer.from_pretrained(mid)
            self.model = AutoModelForCausalLM.from_pretrained(mid, quantization_config=bnb, device_map="auto")
            self.model.eval()
            print(f"Local 4-bit VRAM: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")
        elif live and tier == "strong_frontier":
            from google import genai
            self.client = genai.Client()

    def solve(self, prompt: str, context: str) -> tuple[str, int, int, float, str]:
        if self.live and self.tier == "weak_local" and self.model is not None:
            msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Context:\n{context}\n\nTask:\n{prompt}"}]
            txt = self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = self.tokenizer(txt, return_tensors="pt", truncation=True, max_length=6144).to(self.model.device)
            in_tok = int(inp["input_ids"].shape[1])
            t0 = time.perf_counter()
            with torch.inference_mode():
                out = self.model.generate(**inp, max_new_tokens=220, do_sample=False, pad_token_id=self.tokenizer.eos_token_id)
            torch.cuda.synchronize()
            ms = (time.perf_counter() - t0) * 1000.0
            ans = self.tokenizer.decode(out[0][in_tok:], skip_special_tokens=True).strip()
            return ans, in_tok, int(out.shape[1] - in_tok), ms, "live"

        if self.live and self.tier == "strong_frontier" and self.client is not None:
            from google.genai import types
            full = f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nTask:\n{prompt}"
            t0 = time.perf_counter()
            resp = self.client.models.generate_content(
                model="gemini-3.8-flash",
                contents=full,
                config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=220),
            )
            ms = (time.perf_counter() - t0) * 1000.0
            meta = resp.usage_metadata
            in_tok = int(getattr(meta, "prompt_token_count", 0) or count_tokens(full))
            out_tok = int(getattr(meta, "candidates_token_count", 0) or count_tokens(resp.text or ""))
            return (resp.text or "").strip(), in_tok, out_tok, ms, "live"

        in_tok = count_tokens(SYSTEM_PROMPT) + count_tokens(prompt) + count_tokens(context)
        out_tok = 175
        ms = ((in_tok / 900.0 + out_tok / 32.0) * 1000.0) if self.tier == "weak_local" else (450.0 + (in_tok / 1800.0 + out_tok / 85.0) * 1000.0)
        return f"[calibrated:{self.tier}]", in_tok, out_tok, ms, "calibrated_sim"


weak_solver = SolverBackend("weak_local", live=HAS_GPU)
strong_solver = SolverBackend("strong_frontier", live=HAS_GEMINI)
print(f"Weak Solver  : {'LIVE T4 4-bit' if weak_solver.live else 'calibrated_sim'}")
print(f"Strong Solver: {'LIVE Gemini API' if strong_solver.live else 'calibrated_sim'}")

In [ ]:
# @title 6. Execução End-to-End (Resolver & Mensurar os 4 Pipelines + PGR)
import pandas as pd


def middle_burial_ratio(context_text: str, expected_symbols: list[str]) -> float:
    if not context_text or not expected_symbols:
        return 0.0
    total = max(1, len(context_text))
    found, buried = 0, 0
    for sid in expected_symbols:
        nd = repo_graph.nodes.get(sid)
        if not nd:
            continue
        sig_ln = nd.source.splitlines()[0].strip() if nd.source else nd.signature
        pos = context_text.find(sig_ln)
        if pos < 0:
            pos = context_text.find(f"def {nd.name}(")
        if pos >= 0:
            found += 1
            if 0.30 <= (pos / total) <= 0.70:
                buried += 1
    return (buried / found) if found > 0 else 0.0


def score_resolution(
    bundle: ContextBundle,
    task: EvalTask,
    tier: str,
    answer: str,
    mode: str,
) -> tuple[float, float, float, float, float, float]:
    ret_s, exp_s = set(bundle.retrieved_symbols), set(task.expected_symbols)
    tp = len(ret_s & exp_s)
    prec = tp / len(ret_s) if ret_s else 0.0
    fn_rec = tp / len(exp_s) if exp_s else 0.0
    f1 = (2 * prec * fn_rec / (prec + fn_rec)) if (prec + fn_rec) > 0 else 0.0
    file_rec = len(set(bundle.retrieved_files) & set(task.expected_files)) / len(set(task.expected_files)) if task.expected_files else 0.0
    burial = middle_burial_ratio(bundle.context_text, task.expected_symbols)

    loc_signal = 0.60 * fn_rec + 0.25 * file_rec + 0.15 * min(1.0, prec * 1.25)
    length_stress = min(1.0, max(0.0, (bundle.context_tokens - 1200) / 8000.0))
    retention = 1.0 - 0.28 * burial * length_stress
    if mode == "live":
        ans_low = answer.lower()
        cited = sum(1 for s in task.expected_symbols if repo_graph.nodes[s].name.lower() in ans_low)
        qual = 0.55 * (cited / max(1, len(task.expected_symbols))) + 0.30 * loc_signal + 0.15 * retention
    else:
        tier_factor = 1.00 if tier == "strong_frontier" else (0.94 if task.expected_complexity == "low" else 0.60)
        qual = loc_signal * retention * tier_factor
    return file_rec, fn_rec, prec, f1, burial, round(qual, 4)


def cost_per_1k(tier: str, in_tok: int, out_tok: int, ms: float) -> float:
    if tier == "strong_frontier":
        return 1000.0 * ((in_tok / 1_000_000) * 1.25 + (out_tok / 1_000_000) * 5.00)
    return 1000.0 * ((ms / 1000.0) / 3600.0) * 0.35


eval_rows = []
weak_qs, strong_qs = [], []
sample_inspection = []

for task in EVAL_TASKS:
    for p_name, bundle in [
        ("Brute Force", run_brute_force(task)),
        ("Naive RAG", rag_retriever.retrieve(task.prompt, top_k=5)),
        ("Unbounded AST 2-Hop", run_unbounded_ast(task)),
    ]:
        ans, in_t, out_t, inf_ms, mode = strong_solver.solve(task.prompt, bundle.context_text)
        fr, fnr, pr, f1, bur, q = score_resolution(bundle, task, "strong_frontier", ans, mode)
        eval_rows.append({
            "pipeline": p_name,
            "task_id": task.task_id,
            "expected_complexity": task.expected_complexity,
            "selected_tier": "strong_frontier",
            "p_strong": 1.0,
            "context_tokens": bundle.context_tokens,
            "file_recall": fr,
            "function_recall": fnr,
            "symbol_precision": pr,
            "symbol_f1": f1,
            "middle_burial_pct": bur * 100.0,
            "quality_score": q,
            "prep_ms": bundle.prep_ms,
            "inference_ms": inf_ms,
            "total_ms": bundle.prep_ms + inf_ms,
            "cost_per_1k_usd": cost_per_1k("strong_frontier", in_t, out_t, inf_ms),
            "mode": mode,
        })

    sl, dec = pt_router.route_query(task.prompt, slicer, extractor, token_budget=2200, max_symbols=6, k_hop=2)
    oc_bundle = ContextBundle(sl.context_text, sl.ordered_node_ids, sl.retrieved_files, sl.context_tokens, sl.latency_ms + dec.routing_latency_ms)

    ans_w, in_w, out_w, ms_w, m_w = weak_solver.solve(task.prompt, oc_bundle.context_text)
    ans_s, in_s, out_s, ms_s, m_s = strong_solver.solve(task.prompt, oc_bundle.context_text)
    _, _, _, _, _, qw = score_resolution(oc_bundle, task, "weak_local", ans_w, m_w)
    _, _, _, _, _, qs = score_resolution(oc_bundle, task, "strong_frontier", ans_s, m_s)
    weak_qs.append(qw)
    strong_qs.append(qs)

    use_strong = dec.is_strong
    ans_o, in_o, out_o, ms_o, m_o = (ans_s, in_s, out_s, ms_s, m_s) if use_strong else (ans_w, in_w, out_w, ms_w, m_w)
    fr, fnr, pr, f1, bur, qo = score_resolution(oc_bundle, task, dec.selected_tier, ans_o, m_o)
    eval_rows.append({
        "pipeline": "OccamRoute (ML)",
        "task_id": task.task_id,
        "expected_complexity": task.expected_complexity,
        "selected_tier": dec.selected_tier,
        "p_strong": dec.p_strong,
        "context_tokens": oc_bundle.context_tokens,
        "file_recall": fr,
        "function_recall": fnr,
        "symbol_precision": pr,
        "symbol_f1": f1,
        "middle_burial_pct": bur * 100.0,
        "quality_score": qo,
        "prep_ms": oc_bundle.prep_ms,
        "inference_ms": ms_o,
        "total_ms": oc_bundle.prep_ms + ms_o,
        "cost_per_1k_usd": cost_per_1k(dec.selected_tier, in_o, out_o, ms_o),
        "mode": m_o,
    })
    sample_inspection.append((task, sl, dec, ans_o))

df_eval = pd.DataFrame(eval_rows)
Path("artifacts").mkdir(exist_ok=True)
df_eval.to_csv("artifacts/swebench_eval_results.csv", index=False)

Q_WEAK = sum(weak_qs) / len(weak_qs)
Q_STRONG = sum(strong_qs) / len(strong_qs)
print(f"Avaliação salva em artifacts/swebench_eval_results.csv | Q_weak={Q_WEAK:.3f} | Q_strong={Q_STRONG:.3f}")

In [ ]:
# @title 7. Tabela Comparativa Final (PGR, Localização, Middle-Burial, Custo/1k)
order = ["Brute Force", "Naive RAG", "Unbounded AST 2-Hop", "OccamRoute (ML)"]
summary = (
    df_eval.groupby("pipeline", as_index=False)
    .agg(
        file_recall=("file_recall", "mean"),
        function_recall=("function_recall", "mean"),
        symbol_precision=("symbol_precision", "mean"),
        symbol_f1=("symbol_f1", "mean"),
        middle_burial_pct=("middle_burial_pct", "mean"),
        mean_tokens=("context_tokens", "mean"),
        mean_latency_ms=("total_ms", "mean"),
        cost_per_1k_usd=("cost_per_1k_usd", "mean"),
        quality_score=("quality_score", "mean"),
        local_offload_pct=("selected_tier", lambda s: (s == "weak_local").mean() * 100.0),
    )
)
summary["pipeline"] = pd.Categorical(summary["pipeline"], categories=order, ordered=True)
summary = summary.sort_values("pipeline").reset_index(drop=True)

bf_tok = float(summary.loc[summary["pipeline"] == "Brute Force", "mean_tokens"].iloc[0])
summary["token_savings_pct"] = (1.0 - summary["mean_tokens"] / bf_tok) * 100.0
summary["pgr_pct"] = ((summary["quality_score"] - Q_WEAK) / max(1e-6, Q_STRONG - Q_WEAK)) * 100.0

print(summary.round(3).to_string(index=False))

In [ ]:
# @title 8. Gráfico 1: Fronteira de Pareto & Localização vs. Custo (Dark Theme)
import matplotlib.pyplot as plt

Path("assets").mkdir(exist_ok=True)
BG_FIG, BG_AX, TEXT_MAIN, TEXT_MUTED, GRID_COL = "#0D1117", "#161B22", "#E6EDF3", "#8B949E", "#30363D"
PALETTE = {
    "Brute Force": "#F85149",
    "Naive RAG": "#D29922",
    "Unbounded AST 2-Hop": "#A371F7",
    "OccamRoute (ML)": "#2EA043",
}

plt.rcParams.update({
    "figure.facecolor": BG_FIG,
    "axes.facecolor": BG_AX,
    "axes.edgecolor": GRID_COL,
    "axes.labelcolor": TEXT_MAIN,
    "xtick.color": TEXT_MAIN,
    "ytick.color": TEXT_MUTED,
    "text.color": TEXT_MAIN,
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 5.5), dpi=250)
fig.patch.set_facecolor(BG_FIG)

for _, r in summary.iterrows():
    p = str(r["pipeline"])
    ax1.scatter(r["mean_tokens"], r["quality_score"], color=PALETTE[p], s=220 if "OccamRoute" in p else 140, marker="*" if "OccamRoute" in p else "o", edgecolors="#FFFFFF" if "OccamRoute" in p else "#0D1117", zorder=5)
    ax1.annotate(f"{p}\n{r['mean_tokens']:,.0f} tok | PGR={r['pgr_pct']:.0f}%", (r["mean_tokens"], r["quality_score"]), xytext=(8, 4), textcoords="offset points", fontsize=8.5, color="#7EE787" if "OccamRoute" in p else TEXT_MAIN)

ax1.axhline(Q_STRONG, color="#58A6FF", ls="--", lw=1.0, alpha=0.6)
ax1.axhline(Q_WEAK, color=TEXT_MUTED, ls="--", lw=1.0, alpha=0.6)
ax1.set_xscale("log")
ax1.set_title("Pareto Frontier: Resolution Quality vs. Tokens", fontsize=12, fontweight="bold")
ax1.set_xlabel("Mean Context Tokens (log)")
ax1.set_ylabel("Resolution Quality Score")
ax1.grid(True, color=GRID_COL, ls="--", alpha=0.5)

x = np.arange(len(order))
w = 0.30
ax2.bar(x - w / 2, summary["function_recall"], width=w, color="#58A6FF", label="Function Recall")
ax2.bar(x + w / 2, summary["symbol_f1"], width=w, color="#2EA043", label="Symbol F1")
for idx, (_, r) in enumerate(summary.iterrows()):
    ax2.text(idx, 1.07, f"${r['cost_per_1k_usd']:.2f}/1k\n({r['token_savings_pct']:+.0f}% tok)", ha="center", fontsize=8.3, color="#7EE787" if "OccamRoute" in str(r["pipeline"]) else TEXT_MUTED)

ax2.set_title("Localization F1 & Cost per 1,000 Executions", fontsize=12, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(order, fontsize=9, rotation=10)
ax2.set_ylim(0, 1.24)
ax2.grid(axis="y", color=GRID_COL, ls="--", alpha=0.5)
ax2.legend(facecolor=BG_FIG, edgecolor=GRID_COL, fontsize=8.8, loc="upper left")

for ax in (ax1, ax2):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
p1 = Path("assets/swebench_pareto_frontier.png")
fig.savefig(p1, dpi=250, bbox_inches="tight", facecolor=BG_FIG)
plt.show()
print(f"Salvo: {p1}")

In [ ]:
# @title 9. Gráfico 2: Footprint por Tarefa & Probabilidade P(strong) do Roteador ML
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 5.5), dpi=250)
fig.patch.set_facecolor(BG_FIG)

piv = df_eval.pivot(index="task_id", columns="pipeline", values="context_tokens")[order]
for p in order:
    ax1.plot(piv.index, piv[p], marker="o", lw=2.2 if "OccamRoute" in p else 1.5, label=p, color=PALETTE[p])
ax1.set_yscale("log")
ax1.set_title("Per-Task Token Consumption (Log Scale)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Task ID")
ax1.set_ylabel("Context Tokens")
ax1.tick_params(axis="x", rotation=30)
ax1.grid(True, color=GRID_COL, ls="--", alpha=0.5)
ax1.legend(facecolor=BG_FIG, edgecolor=GRID_COL, fontsize=8.3, loc="lower right")

oc_rows = df_eval[df_eval["pipeline"] == "OccamRoute (ML)"]
colors_t = ["#58A6FF" if c == "low" else "#F78166" for c in oc_rows["expected_complexity"]]
ax2.bar(oc_rows["task_id"], oc_rows["p_strong"], color=colors_t, edgecolor="#0D1117")
ax2.axhline(optimal_tau, color="#2EA043", ls="--", lw=1.6, label=f"Threshold tau={optimal_tau:.2f}")
ax2.set_title("Trained ML Router P(strong | x) per Task", fontsize=12, fontweight="bold")
ax2.set_xlabel("Task ID (Blue=Low, Coral=High)")
ax2.set_ylabel("Predicted P(strong)")
ax2.set_ylim(0, 1.08)
ax2.tick_params(axis="x", rotation=30)
ax2.grid(axis="y", color=GRID_COL, ls="--", alpha=0.5)
ax2.legend(facecolor=BG_FIG, edgecolor=GRID_COL, fontsize=8.8)

for ax in (ax1, ax2):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
p2 = Path("assets/swebench_task_breakdown.png")
fig.savefig(p2, dpi=250, bbox_inches="tight", facecolor=BG_FIG)
plt.show()
print(f"Salvo: {p2}")

In [ ]:
# @title 10. Inspeção Detalhada de Uma Tarefa Resolvida pelo OccamRoute
task_ex, sl_ex, dec_ex, ans_ex = sample_inspection[-1]
print(f"Task ID          : {task_ex.task_id} ({task_ex.repo})")
print(f"Prompt           : {task_ex.prompt}")
print(f"Símbolos U-Order : {sl_ex.ordered_node_ids}")
print(f"Tokens / Compr.  : {sl_ex.context_tokens} tokens ({sl_ex.compression_ratio * 100:.1f}% economia)")
print(f"Roteamento ML    : tier={dec_ex.selected_tier} | P(strong)={dec_ex.p_strong:.3f} | {dec_ex.routing_latency_ms:.2f} ms")
print(f"Gabarito         : {task_ex.expected_symbols}")